In [6]:
"""Notebook helper to fill the Gibson template with part names.

How to use this notebook:
1. Make sure `gibson_designs_template.xlsx` is closed in Excel.
2. If needed, change the settings below.
3. Run the next cell.
"""

from pathlib import Path

from openpyxl import load_workbook

SUPPORTED_SUFFIXES = {".gb", ".gbk", ".genbank", ".dna"}
PART_COLUMNS = {
    "01_": "A",
    "02_": "B",
    "03_": "C",
    "04_": "D",
    "05_": "E",
    "06_": "F",
}

# Update these settings only if your notebook runs from a different folder.
BASE_DIR = Path.cwd()
TEMPLATE_PATH = BASE_DIR / "gibson_designs_template.xlsx"
SHEET_NAME = "Info"
START_ROW = 3


In [7]:
def extract_part_name(genbank_path: Path) -> str:
    """Return the filename without the file extension."""

    return genbank_path.stem


def find_folder(base_dir: Path, prefix: str) -> Path:
    """Find the single folder whose name starts with the given prefix."""

    matches = [path for path in sorted(base_dir.glob(f"{prefix}*")) if path.is_dir()]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one folder starting with {prefix!r}, found: "
            f"{[path.name for path in matches]}"
        )
    return matches[0]


def collect_part_names(base_dir: Path) -> dict[str, list[str]]:
    """Collect part names for Excel columns A-F."""

    part_names = {}
    for prefix, column in PART_COLUMNS.items():
        folder = find_folder(base_dir, prefix)
        files = sorted(
            path for path in folder.iterdir() if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES
        )
        part_names[column] = [extract_part_name(path) for path in files]
    return part_names


def write_parts_to_template(template_path: Path, part_names: dict[str, list[str]]) -> None:
    """Write the collected names into the Excel template."""

    workbook = load_workbook(template_path)
    sheet = workbook[SHEET_NAME]

    for column, names in part_names.items():
        for row in range(START_ROW, sheet.max_row + 1):
            sheet[f"{column}{row}"] = None
        for offset, name in enumerate(names):
            sheet[f"{column}{START_ROW + offset}"] = name

    try:
        workbook.save(template_path)
    except PermissionError as error:
        raise PermissionError(
            f"Could not save {template_path.name}. Close the Excel file first and run the cell again."
        ) from error


if not BASE_DIR.exists():
    raise FileNotFoundError(f"Base directory does not exist: {BASE_DIR}")
if not TEMPLATE_PATH.exists():
    raise FileNotFoundError(f"Template file does not exist: {TEMPLATE_PATH}")

part_names = collect_part_names(BASE_DIR)

print("Preview of names to be written:")
for column, names in part_names.items():
    print(f"Column {column}: {names}")

write_parts_to_template(TEMPLATE_PATH, part_names)

print(f"Updated workbook: {TEMPLATE_PATH}")
for column, names in part_names.items():
    print(f"Column {column}: wrote {len(names)} part name(s)")

for column in sorted(part_names):
    if part_names[column] == []:
        print(f"Note: column {column} stayed empty because no supported files were found in the corresponding folder.")


Preview of names to be written:
Column A: ['PH36', 'Psyn', 'Ptuf']
Column B: ['RBS_01']
Column C: ['GFP', 'RFP_CD', 'YFP_CD']
Column D: ['TrrnB']
Column E: ['UNS1_UNS4_BB', 'UNS4_UNS5_BB', 'UNS5_UNS6_BB', 'UNS6_UNS7_BB', 'UNS7_UNS10_BB']
Column F: ['pEVmC(K)_AE']
Updated workbook: c:\Users\koesters\Sciebo\Git\Repositories\assembly_designer\examples\02 insilico Plasmid Construction\05 3G Assembly\01_Simple_Assembly_Example\gibson_designs_template.xlsx
Column A: wrote 3 part name(s)
Column B: wrote 1 part name(s)
Column C: wrote 3 part name(s)
Column D: wrote 1 part name(s)
Column E: wrote 5 part name(s)
Column F: wrote 1 part name(s)
